# 15 — Past-only defensive motion vectors

**Aim:** Add directional movement information to our verified static snapshots for **all 20 matches**, using exact past tracking frames only. This is a necessary ingredient for a future *scenario-based* defender-reach model, **not** a validated interception model.

**Inputs:** Notebook 05's 20 `P1_canonical_decisions_match_*_ROLES.jsonl` files, Notebook 14's `P3_all_matches_motion_gate.json`, and original SkillCorner tracking files.

**Output:** `outputs/P3_motion_vectors_match_<match_id>.jsonl` locally; upload only `outputs/P3_motion_vectors_gate.json`.

For each defender in each canonical possession snapshot we estimate a finite-difference **velocity vector** from `(t₀−5, t₀)`, in m/s using a nominal 10Hz sampling rate. We also retain a 1-second estimate and a *pre-decision-only* `(t₀−10, t₀−5)` estimate. The latter is useful for checking direction stability; no future frame enters any feature. Keep measured displacement separate from acceleration limits, turning abilities and reaction-time assumptions.


In [1]:
from pathlib import Path
from collections import Counter
from datetime import datetime,timezone
import json,math,statistics

ROOT=next((p for p in (Path.cwd().resolve(),*Path.cwd().resolve().parents) if (p/'data'/'matches').is_dir()),None)
if ROOT is None:raise FileNotFoundError('Run inside SkillCorner repository')
OUT=ROOT/'outputs'; TRACK=ROOT/'data'/'matches'
source=json.loads((OUT/'P3_all_matches_motion_gate.json').read_text())
mids=sorted(str(x['match_id']) for x in source['matches'])
assert len(mids)==20 and source['decisions']==18792
print('Matches:',len(mids))


Matches: 20


## 1 — Validated vector helper

Frames are addressed by **(period, absolute frame)** to avoid crossing halves. Vector components are tracking-coordinate X/Y; we do not equate a player's recent direction of travel with their possible future movement. Endpoint detection quality is explicit.

For zero or missing displacement, direction of motion is **undefined**; no unit heading is manufactured.


In [2]:
def parse_int(x):
    try:return int(x)
    except (ValueError,TypeError):return None

def finite_xy(d):
    if not isinstance(d,dict):return None
    try:
        x,y=float(d['x']),float(d['y'])
        return (x,y) if math.isfinite(x) and math.isfinite(y) else None
    except (KeyError,TypeError,ValueError):return None

def index_players(frame):
    if frame is None:return {}
    return {str(p['player_id']):p for p in (frame.get('player_data') or []) if p.get('player_id') is not None}

def vector(a,b,dt):
    pa,pb=finite_xy(a),finite_xy(b)
    if pa is None or pb is None:return None
    return [(pa[0]-pb[0])/dt,(pa[1]-pb[1])/dt]

def magnitude(v):
    return math.hypot(*v) if v is not None else None

def detected(a,b):
    return bool(a is not None and b is not None and
                a.get('is_detected') is True and b.get('is_detected') is True)

def alignment(a,b):
    if a is None or b is None:return None
    na,nb=magnitude(a),magnitude(b)
    if na==0 or nb==0:return None
    return max(-1.0,min(1.0,(a[0]*b[0]+a[1]*b[1])/(na*nb)))


## 2 — Produce one decision record with its defensive movement vectors

We use the existing canonical opponent identities, so we do **not** infer defender teams from geometry or future events. Current and historical coordinates are read directly from source tracking.

The reported 15 m/s exceedances are **flags for inspection**, not automatic exclusion rules. Detection-quality sensitivity remains the intended primary comparison.


In [3]:
def process_match(mid):
    inp=OUT/f'P1_canonical_decisions_match_{mid}_ROLES.jsonl'
    decisions=[json.loads(s) for s in inp.open(encoding='utf-8') if s.strip()]
    keys={(int(d['period']),int(d['frame'])-h) for d in decisions for h in (0,5,10) if int(d['frame'])>=h}
    frames={}
    with (TRACK/mid/f'{mid}_tracking_extrapolated.jsonl').open(encoding='utf-8') as f:
        for line in f:
            z=json.loads(line)
            p,t=parse_int(z.get('period')),parse_int(z.get('frame'))
            if p is None or t is None:continue
            if (p,t) in keys:
                if (p,t) in frames:raise AssertionError(f'{mid}: duplicate requested {(p,t)}')
                frames[(p,t)]=z
    counts=Counter(); anomalies=[]
    output=OUT/f'P3_motion_vectors_match_{mid}.jsonl'
    with output.open('w',encoding='utf-8') as f:
        for d in decisions:
            p,t=int(d['period']),int(d['frame'])
            f0=frames.get((p,t))
            if f0 is None:raise AssertionError(f'{mid}: missing present frame {(p,t)}')
            f5=frames.get((p,t-5)) if t>=5 else None
            f10=frames.get((p,t-10)) if t>=10 else None
            p0,p5,p10=index_players(f0),index_players(f5),index_players(f10)
            defender_ids=[str(x['player_id']) for q in ('detected','extrapolated') for x in d.get(f'opponents_{q}',[])]
            if len(defender_ids)!=len(set(defender_ids)):raise AssertionError(f'{mid}: duplicate defender')
            defs=[]
            for pid in defender_ids:
                a,b,c=p0.get(pid),p5.get(pid),p10.get(pid)
                v05=vector(a,b,0.5)
                v010=vector(a,c,1.0)
                v105=vector(b,c,0.5)
                speed=magnitude(v05)
                if speed is not None:
                    counts['defender_05_finite']+=1
                    if detected(a,b):counts['defender_05_both_detected']+=1
                    if speed>15:
                        counts['defender_05_gt_15']+=1
                        anomalies.append({'period':p,'frame':t,'defender_id':pid,
                                          'speed_mps':speed,'both_detected':detected(a,b)})
                counts['defender_10_finite']+=v010 is not None
                defs.append({'player_id':pid,'velocity_0_to_minus5_mps':v05,
                             'velocity_0_to_minus10_mps':v010,
                             'velocity_minus5_to_minus10_mps':v105,
                             'past_halfsecond_speed_mps':speed,
                             'heading_stability_cosine':alignment(v05,v105),
                             't0_detected':a is not None and a.get('is_detected') is True,
                             'halfsec_both_detected':detected(a,b),
                             'fullsec_both_detected':detected(a,c)})
            record={'match_id':mid,'period':p,'frame':t,
                    'possession_event_id':d['possession_event_id'],
                    'attacking_team_id':d['attacking_team_id'],
                    'carrier_player_id':d['carrier_player_id'],
                    'defenders':defs,
                    'past_frame_minus5_available':f5 is not None,
                    'past_frame_minus10_available':f10 is not None,
                    'interpretation':'retrospective_velocity_not_maximum_reach'}
            f.write(json.dumps(record,separators=(',',':'))+'\n')
            counts['decisions']+=1
            counts['defender_records']+=len(defs)
            counts['past_5_frame_present']+=f5 is not None
            counts['past_10_frame_present']+=f10 is not None
    assert counts['decisions']==len(decisions)
    # Cross-check against independent Notebook 14 report.
    old=next(z for z in source['matches'] if str(z['match_id'])==mid)['counts']
    checks={'defender_05_finite':'defender_5_finite',
            'defender_05_both_detected':'defender_5_detected_endpoints',
            'defender_10_finite':'defender_10_finite',
            'defender_records':'defender_records',
            'past_5_frame_present':'past_5_frame_present',
            'past_10_frame_present':'past_10_frame_present'}
    for new,oldkey in checks.items():
        if counts[new]!=old.get(oldkey,0):
            raise AssertionError(f'{mid}: {new} differs from Notebook 14 ({counts[new]} vs {old.get(oldkey,0)})')
    return {'match_id':mid,'counts':dict(counts),'anomaly_examples':anomalies[:10]}


## 3 — Execute and cross-check all 20 matches

Only past frames enter the velocity estimates. Agreement with Notebook 14's independent displacement calculations is required; if a match disagrees, stop rather than quietly proceeding.


In [4]:
results=[]
for mid in mids:
    r=process_match(mid)
    results.append(r)
    print(mid,r['counts']['decisions'],'decisions;',
          r['counts'].get('defender_05_finite',0),'half-second vectors;',
          r['counts'].get('defender_05_gt_15',0),'anomaly flags')


1874553 959 decisions; 10395 half-second vectors; 0 anomaly flags
1886347 999 decisions; 10901 half-second vectors; 0 anomaly flags
1899585 929 decisions; 10076 half-second vectors; 0 anomaly flags
1925299 1062 decisions; 11649 half-second vectors; 0 anomaly flags
1927964 1140 decisions; 12353 half-second vectors; 0 anomaly flags
1953632 979 decisions; 10571 half-second vectors; 0 anomaly flags
1959846 946 decisions; 10252 half-second vectors; 0 anomaly flags
1986691 915 decisions; 9900 half-second vectors; 0 anomaly flags
1996435 1005 decisions; 10879 half-second vectors; 0 anomaly flags
1996436 786 decisions; 8580 half-second vectors; 0 anomaly flags
2006229 979 decisions; 10670 half-second vectors; 0 anomaly flags
2006363 976 decisions; 10571 half-second vectors; 0 anomaly flags
2007448 863 decisions; 9394 half-second vectors; 1 anomaly flags
2007721 959 decisions; 10483 half-second vectors; 0 anomaly flags
2010085 935 decisions; 10197 half-second vectors; 0 anomaly flags
2011166 75

In [5]:
totals=Counter()
for r in results:totals.update(r['counts'])
assert totals['decisions']==18792
assert totals['defender_05_finite']==source['counts']['defender_5_finite']
assert totals['defender_05_both_detected']==source['counts']['defender_5_detected_endpoints']
report={'created_utc':datetime.now(timezone.utc).isoformat(),
 'purpose':'Past-only defender velocity vectors and quality gating; no interception assumptions',
 'matches':20,'totals':dict(totals),'per_match':results,
 'source_gate_agreement':True,
 'limitations':[
 'Finite differences at nominal 10 Hz, never forward frames',
 'History conditioned on possession starts; sampling is not representative of all match play',
 'Velocity estimates are retrospective, not maximum running ability or response to pressure',
 'Two detected endpoints do not certify all intermediate frames',
 'Missing historical tracking yields null, not imputation',
 'No defender reach, ball-speed assumptions, pass feasibility or carrying opportunity labels',
 'No causal or predictive inference; player/decision/match dependence persists']}
gate=OUT/'P3_motion_vectors_gate.json'
gate.write_text(json.dumps(report,indent=2),encoding='utf-8')
print(json.dumps({'matches':20,'totals':dict(totals)},indent=2))
print('Upload:',gate)


{
  "matches": 20,
  "totals": {
    "defender_05_finite": 204127,
    "defender_05_both_detected": 130769,
    "defender_10_finite": 203170,
    "decisions": 18792,
    "defender_records": 206338,
    "past_5_frame_present": 18778,
    "past_10_frame_present": 18776,
    "defender_05_gt_15": 1
  }
}
Upload: /Users/fadihaddad/Documents/GitHub/SkillCorner_opendata_soccer/outputs/P3_motion_vectors_gate.json
